In [ ]:
# CELL 1 - Load feature dataset + prep
import os
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/time_window_features.csv")
df["TransactionDT"] = pd.to_datetime(df["TransactionDT"])

os.makedirs("../reports", exist_ok=True)  # savefig fails if folder is missing

print(f"Loaded: {df.shape}")
df.head()

In [ ]:
# CELL 2 - Velocity risk (defined ONCE, safe denominator)
# Share of the day's prior transactions concentrated in the last hour.
# clip(lower=1) prevents division by zero for first-time transactions
# (txn_count_24h == 0), which now exist in the dataset.
df["velocity_risk"] = (
    df["txn_count_1h"] / df["txn_count_24h"].clip(lower=1)
).clip(0, 1)

In [ ]:
# CELL 3 - Class split + quick stats
fraud = df[df["isFraud"] == 1]
non_fraud = df[df["isFraud"] == 0]

print(df[["txn_count_1h", "txn_count_24h", "velocity_risk"]].describe().round(3))
print(f"\nFraud:     {len(fraud):,} ({len(fraud) / len(df):.2%})")
print(f"Non-fraud: {len(non_fraud):,}")

In [ ]:
# CELL 4 - Plot helper (density + legend + safe save)
def plot_fraud_vs_legit(column, title, xlabel, bins=50, xlim=None, save_as=None):
    """Overlaid fraud vs non-fraud histograms.

    density=True is essential: fraud is ~3.5% of rows, so raw-count
    plots make the fraud distribution invisible. savefig runs BEFORE
    plt.show(), because show() clears the figure (blank-PNG bug).
    """
    plt.figure(figsize=(8, 4.5))
    plt.hist(non_fraud[column].dropna(), bins=bins, alpha=0.6,
             density=True, label="Non-fraud")
    plt.hist(fraud[column].dropna(), bins=bins, alpha=0.6,
             density=True, label="Fraud")
    if xlim:
        plt.xlim(*xlim)
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    if save_as:
        plt.savefig(save_as, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_as}")
    plt.show()

In [ ]:
# CELL 5 - Velocity distributions
plot_fraud_vs_legit("txn_count_1h",
                    "Txn Count (1h): Fraud vs Non-Fraud",
                    "Prior transactions in last 1 hour")

plot_fraud_vs_legit("txn_count_24h",
                    "Txn Count (24h): Fraud vs Non-Fraud",
                    "Prior transactions in last 24 hours")

In [ ]:
# CELL 6 - Spending behavior distributions
plot_fraud_vs_legit("avg_amt_24h",
                    "Average Amount (24h): Fraud vs Non-Fraud",
                    "Avg prior transaction amount (24h)",
                    xlim=(0, 500))   # right tail runs to ~4900; clip for readability

plot_fraud_vs_legit("amount_zscore_24h",
                    "Amount Deviation (Z-Score): Fraud vs Non-Fraud",
                    "Amount z-score vs 24h prior baseline",
                    bins=100, xlim=(-10, 10))

In [ ]:
# CELL 7 - Velocity risk distribution (saved for README/report)
plot_fraud_vs_legit("velocity_risk",
                    "Velocity Risk: Fraud vs Non-Fraud",
                    "Velocity risk score",
                    save_as="../reports/velocity_vs_fraud.png")

In [ ]:
# CELL 8 - Night-time fraud rate
night_fraud_rate = df.groupby("is_night_txn")["isFraud"].mean()
print(night_fraud_rate.round(4))

ax = night_fraud_rate.plot(kind="bar",
                           title="Fraud Rate: Night (00-04) vs Day")
ax.set_xticklabels(["Day", "Night"], rotation=0)
plt.ylabel("Fraud rate")
plt.savefig("../reports/night_vs_day_fraud_rate.png", dpi=150,
            bbox_inches="tight")   # BEFORE show()
plt.show()

In [ ]:
# CELL 9 - Summary table: feature means by class
# (works with old AND new feature CSVs - checks which columns exist)
candidates = [
    "txn_count_1h", "txn_count_24h", "velocity_risk",
    "avg_amt_24h", "amount_zscore_24h", "is_night_txn",
    "no_prior_24h", "single_prior_24h",
]
check_cols = [c for c in candidates if c in df.columns]

summary = df.groupby("isFraud")[check_cols].mean().round(3)
summary.index = ["Non-fraud", "Fraud"]
print(summary)

In [ ]:
# CELL 10 - Save dataset WITH velocity_risk for modeling notebooks
df.to_csv("../data/processed/time_window_features.csv", index=False)
print(f"Saved with velocity_risk: {df.shape}")